# Практическая работа 13 (Тест)
## Тема: Подготовка факторов модели

### Настройка окружения

In [1]:
import pandas as pd
import numpy as np

Дан набор данных [auto.csv](./auto.csv), содержащий информацию о характеристиках подержанных автомобилей. Подробно изучить описание атрибутов можно по [ссылке](https://archive.ics.uci.edu/dataset/10/automobile) в источнике.

In [2]:
auto = pd.read_csv("auto.csv")
auto.head()

,symboling,normalized_losses,make,fuel_type,aspiration,num_doors,body_style,drive_wheels,engine_location,wheel_base,...,engine_size,fuel_system,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price
0,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0
1,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0
2,1,NaN,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500.0
3,2,164.0,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950.0
4,2,164.0,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450.0


### Вопрос 1
Определите наличие пропущенных значений. В качестве ответа укажите кол-во пропусков в столбце **peak_rpm**.

In [5]:
auto["horsepower"].isna().sum()

np.int64(2)

Выполним обработку пропущенных значений:

1. Удалим столбец **normalized_losses**.

In [6]:
auto = auto.drop(columns=["normalized_losses"])

2. Заполним пропуски в столбце **num_doors** самым часто встречающимся значением.

In [7]:
most_common_doors = auto["num_doors"].mode()[0]

auto["num_doors"] = auto["num_doors"].fillna(most_common_doors)

3. Заполним пропуски в столбцах **bore** и **stroke** средним значением, округленным до 2 знаков после запятой.

In [8]:
for col in ["bore", "stroke"]:
    col_mean = auto[col].mean()
    auto[col] = auto[col].fillna(round(col_mean, 2))

4. Заполним пропуски в столбцах **horsepower**, **peak_rpm** и **price** средним значением, округленным до целого.

In [9]:
for col in ["horsepower", "peak_rpm", "price"]:
    col_mean = auto[col].mean()
    auto[col] = auto[col].fillna(round(col_mean))

### Вопрос 2
Укажите кол-во категориальных столбцов.

In [12]:
num_categorical = auto.select_dtypes(include=["object", "category"]).shape[1]

num_categorical

/var/folders/fs/l607bnnd1cgg08pttn0yqbyh0000gn/T/ipykernel_29345/825704773.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  num_categorical = auto.select_dtypes(include=["object", "category"]).shape[1]


10

### Вопрос 3
В нашем наборе данных есть два столбца данных, значения которых представляют собой слова, используемые для представления чисел:

1. количество цилиндров в двигателе (**num_cylinders**);
2. количество дверей в машине (**num_doors**).

С помощью метода `replace` выполните замену текстовых значений их числовыми эквивалентами.

В качестве ответа укажите кол-во автомобилей с шестью цилиндрами в двигателе.

In [16]:
num_map = {
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "eight": 8,
    "twelve": 12,
}

pd.set_option("future.no_silent_downcasting", True)

auto[["num_cylinders", "num_doors"]] = (
    auto[["num_cylinders", "num_doors"]].replace(num_map).infer_objects(copy=False)
)

count_six_cylinders = (auto["num_cylinders"] == 12).sum()

count_six_cylinders

/var/folders/fs/l607bnnd1cgg08pttn0yqbyh0000gn/T/ipykernel_29345/2974273976.py:11: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option("future.no_silent_downcasting", True)
/var/folders/fs/l607bnnd1cgg08pttn0yqbyh0000gn/T/ipykernel_29345/2974273976.py:14: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  auto[["num_cylinders", "num_doors"]].replace(num_map).infer_objects(copy=False)


np.int64(1)

### Вопрос 4
Для столбца **body_style** выполним кодирование меток (*label encoding*), преобразовав каждое значение в столбце в число.

Для этого:

1. преобразуем столбец в категорию с помощью функции `astype('category')`;

In [17]:
auto["body_style"] = auto["body_style"].astype("category")

2. создадим новый столбец **body_style_cat**, содержащий закодированные значения столбца **body_style** с помощью метода доступа `cat.codes`.

In [18]:
auto["body_style_cat"] = auto["body_style"].cat.codes

В качестве ответа укажите какой код был присвоен типу кузова хэтчбек (*hatchback*).

In [19]:
hatchback_code = auto["body_style"].cat.categories.get_loc("wagon")

hatchback_code

4

### Вопрос 5
С помощью функции `pd.get_dummies()` выполните унитарное кодирование (*One Hot Encoding*) столбцов **body_style** и **drive_wheels**.

В качестве ответа выберите вектор, кодирующий информацию о типе кузова для 72-й машины: [*body_convertible*, *body_hardtop*, *body_hatchback*, *body_sedan*, *body_wagon*].

In [20]:
auto = pd.get_dummies(
    auto, columns=["body_style", "drive_wheels"], prefix=["body", "drive"]
)

encoded_vector = auto.iloc[72][
    [
        "body_convertible",
        "body_hardtop",
        "body_hatchback",
        "body_sedan",
        "body_wagon",
    ]
].values

encoded_vector.astype(int)

array([1, 0, 0, 0, 0])

### Вопрос 6
В нашем наборе данных есть столбец с именем **engine_type** (тип двигателя), который содержит несколько разных значений.

Например, нам для анализа важно, оснащен ли двигатель верхним распредвалом (*Overhead Cam, OHC*) или нет. Другими словами, разные версии *OHC* одинаковы для этого анализа. С помощью метода доступа `str` и функции `np.where` создайте новый столбец **OHC_Code**, который указывает, есть ли в автомобиле двигатель *OHC*.

In [21]:
auto["OHC_Code"] = np.where(auto["engine_type"].str.contains("ohc", case=False), 1, 0)

### Вопрос 7
Выполните целевое кодирование марок автомобилей (**make**). Для этого создайте столбец **make_m_enc**, в котором каждая марка кодируется средним значением целевой переменной - **price**, округленным до целого.

В качестве ответа укажите результат кодирование для марки *renault*.

In [22]:
make_price_mean = auto.groupby("make")["price"].mean().round()

auto["make_m_enc"] = auto["make"].map(make_price_mean)

auto.loc[auto["make"] == "porsche", "make_m_enc"].iloc[0]

np.float64(27762.0)

### Вопрос 8
С помощью библиотеки `sklearn` выполните прямое кодирование (*Label Encoding*) значений столбца **fuel_system**. Результат кодирования сохраните в столбце **fuel_system_code**. В качестве ответа укажите, какой код был присвоен топливной системе *mfi*.

In [23]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

auto["fuel_system_code"] = le.fit_transform(auto["fuel_system"])

mfi_code = le.transform(["mfi"])[0]

mfi_code

np.int64(4)

### Вопрос 9
Выполните *One-Hot* кодирование данных столбцов **fuel_type**, **aspiration**, **engine_location** с помощью библиотеки `sklearn`. Добавьте закодированные столбцы к исходному датафрейму по шаблону: **fuel_type_category1**, **fuel_type_category2** и т.д. В качестве ответа укажите кол-во новых столбцов.

In [24]:
from sklearn.preprocessing import OneHotEncoder

cols_to_encode = ["fuel_type", "aspiration", "engine_location"]

ohe = OneHotEncoder(sparse_output=False, dtype=int)

encoded = ohe.fit_transform(auto[cols_to_encode])

encoded_cols = ohe.get_feature_names_out(cols_to_encode)
encoded_df = pd.DataFrame(encoded, columns=encoded_cols)

auto = auto.join(encoded_df)

encoded_df.shape[1]

6

Теперь мы с вами выполнили кодирование всех категориальных факторов. Для продолжения анализа сохраним в новый датафрейм *auto_model* данные всех столбцов, кроме исходных **make**, **fuel_type**, **aspiration**, **engine_location**, **engine_type**, **fuel_system**.

In [25]:
drop_cols = [
    "make",
    "fuel_type",
    "aspiration",
    "engine_location",
    "engine_type",
    "fuel_system",
]

auto_model = auto.drop(columns=drop_cols)

### Вопрос 10
Выполним стандартизацию всех факторов, на основе которых будем предсказывать значение цены, и сохраним результат в переменную *auto_zscore*:

In [26]:
from sklearn.preprocessing import StandardScaler

auto_zscore = auto_model.copy()
del auto_zscore["price"]

auto_zscore = pd.DataFrame(
    StandardScaler().fit_transform(auto_zscore), columns=auto_zscore.columns
)

В качестве ответа укажите стандартизированное значение длины для 4-го автомобиля (нумерация начинается с 0), округлённое до 2 знаков после запятой.

In [33]:
round(auto_zscore.iloc[203]["height"], 2)

np.float64(0.73)

### Вопрос 11
Выполните разбиение набора данных на обучающую и тестовую выборки:

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    auto_zscore, auto_model.price, test_size=0.2, random_state=42
)

После этого создайте модель линейной регрессии, где в качестве предикторов выступают все переменные, целевой переменной является цена автомобиля. В качестве ответа укажите коэффициент перед значением параметра **symboling**, округлённый до 2 знаков после запятой.

In [35]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

symboling_coeff = model.coef_[X_train.columns.get_loc("width")]

round(symboling_coeff, 2)

np.float64(1272.46)

### Вопрос 12
Оцените качество полученной модели с использованием метрики *MSE*. В качестве ответа укажите получившееся значение, округленное до 2 знаков после запятой.

In [36]:
from sklearn.metrics import mean_squared_error

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

round(mse, 2)

13983699.97

### Вопрос 13

1. Выполним отбор факторов, оставив только 8 первых наиболее коррелирующих по абсолютному значению с ценой. 

In [37]:
corr_matrix = auto_model.corr()

price_corr = corr_matrix["price"].abs()

top_8_corr = price_corr.sort_values(ascending=False).head(9)

top_8_corr = top_8_corr.drop("price")

top_8_columns = top_8_corr.index

2. Выполним построение линейной модели с использованием данных факторов.

In [39]:
X_train_selected = X_train[top_8_columns]
X_test_selected = X_test[top_8_columns]

model = LinearRegression()
model.fit(X_train_selected, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


В качестве ответа укажите значение корреляции между ценой (**price**) и параметром **curb_weight**, округленное до 2 знаков после запятой.

In [40]:
corr_curb_weight = corr_matrix["price"]["city_mpg"]
round(corr_curb_weight, 2)

np.float64(-0.67)

### Вопрос 14
Оцените качество второй модели с использованием метрики *MAE*. В качестве ответа укажите получившееся значение, округленное до 2 знаков после запятой.

In [41]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

y_pred = model.predict(X_test_selected)

mae = root_mean_squared_error(y_test, y_pred)

round(mae, 2)

3310.53

### Вопрос 15
Выберите выводы, которые можно сделать на основе проведенного анализа.

*Варианты ответа:*

1. унитарное кодирование (*One Hot Encoding*) заключается в том, что каждой категории сопоставляется некоторое числовое значение - код;

2. качество второй модели, основанной только на наиболее коррелирующих факторах с целевой переменной, выше, чем первой;

3. недостатком кодирования меток (*label encoding*) является то, что числовые значения могут быть "неверно интерпретированы" алгоритмами;

4. метод стандартизации масштабирует значения переменных так, чтобы они находились в диапазоне от 0 до 1.

*Ответ:* 2, 3